# CASE and TARS experiments

Run two independent experiments. CASE trains and checks a tiny hybrid sequence model. TARS builds a reproducible plan, runs a local mathematical tool, and audits a structured result. Each produces its own evidence. The notebook does not connect CASE to the TARS task API.

**Run locally:** use Python 3.11+ in a Jupyter kernel with PyTorch 2.6+ and select **Restart Kernel and Run All Cells**. The notebook uses CPU execution with synthetic data and no network.

### CASE model snapshot

The embedded files reflect the workspace's `CASE/` model at the file hashes shown in the cells. The notebook flattens the relative imports, omits module headers and selects the CPU scan by setting `HAS_MAMBA_SSM = False`. CASE combines recurrent mixing, sequence-level routing, compressed attention, learned memory, expert dispatch and auxiliary heads. Names used by the original source do not establish equivalence with externally developed architectures. Scope and test expectations draw on the project architecture review, sections 7 and 8. CASE's optional ANN adapter is separate from the forward path used here.

In [1]:
import math
import platform
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 27
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print({"python": platform.python_version(), "torch": torch.__version__, "device": "cpu", "seed": SEED})

{'python': '3.11.0rc2', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 27}


In [2]:
# Source: CASE/config.py
# SHA-256 of original file: ced40571bde0385937915b208e838d1f7ba09f46c53b2c078f418daf314355a8

from __future__ import annotations

import json

from dataclasses import asdict, dataclass, field

from pathlib import Path

from typing import Any, Dict, List, Optional, Union

@dataclass
class CASEConfig:
    """Complete CASE-0.1 Model & Training Configuration."""

    # ─── Vocabulary & Core Geometry ─────────────────────────────────────────
    vocab_size: int = 128_256
    d_model: int = 1536
    n_layers: int = 24
    max_seq_len: int = 4096
    dropout: float = 0.0
    norm_eps: float = 1e-6
    rope_theta: float = 500_000.0

    # ─── Attention Topology (DeepSeek-V3 MLA) ─────────────────────────────────
    attn_type: str = "mla"  # "mla" | "deltanet" | "gqa"
    n_heads: int = 16
    kv_lora_rank: int = 256          # d_c: low-rank KV latent dimension
    q_lora_rank: int = 768           # d_c': low-rank query latent dimension
    qk_nope_head_dim: int = 128      # d_h: content key/query head dimension
    qk_rope_head_dim: int = 64       # d_R: decoupled RoPE rotary head dimension
    v_head_dim: int = 128            # d_v: value head dimension

    # ─── Google Titans Test-Time Neural Memory (LMM) ────────────────────────
    use_titans_memory: bool = True
    titans_mode: str = "mag"         # "mag" (Memory as Gate) | "mac" (Memory as Context) | "mal" (Memory as Layer)
    titans_d_mem: int = 256          # Dimension of neural associative memory keys/values
    titans_chunk_size: int = 64      # Chunk size for parallel associative scan
    titans_lr_init: float = 0.1      # Initial learning rate for surprise gradient updates
    titans_decay_init: float = 0.05  # Initial forgetting factor for memory decay

    # ─── Mamba-2 State Space Duality (SSD) ──────────────────────────────────
    mamba_d_state: int = 64          # N: SSM state dimension per head
    mamba_d_conv: int = 4            # Causal depthwise 1D conv kernel width
    mamba_expand: int = 2            # Inner expansion: d_inner = expand * d_model
    mamba_n_heads: int = 32          # H: SSM heads
    mamba_dt_min: float = 0.001
    mamba_dt_max: float = 0.1
    mamba_conv_bias: bool = True

    # ─── Mixture of Depths (MoD) ────────────────────────────────────────────
    mod_capacity: float = 0.5        # Fraction of tokens routed through Mamba layers (50% skip)
    mod_apply_to_attn: bool = False  # Keep attention globally active by default

    # ─── DeepSeek-V3 Fine-Grained MoE ───────────────────────────────────────
    n_experts: int = 64              # E: Total routed experts
    n_active_experts: int = 4        # K: Top-k active routed experts per token
    n_shared_experts: int = 2        # Always-active shared experts (dense anchor)
    moe_d_ff: int = 1024             # SwiGLU intermediate dimension per expert
    moe_bias_lr: float = 1e-3        # Online bias update learning rate
    moe_router_jitter_noise: float = 0.01
    moe_capacity_factor: float = 1.25

    # ─── GPT-6 Astra Looped Latent Reasoning Core ───────────────────────────
    enable_looped_core: bool = True
    max_latent_loops: int = 1        # 1 for standard pretraining, up to 4 for deep reasoning
    ponder_cost_weight: float = 0.01 # Loss penalty on internal thought cycles

    # ─── Claude Fable / Mythos Process Verifier Head ─────────────────────────
    enable_process_verifier: bool = True
    verifier_loss_weight: float = 0.05

    # ─── Multi-Token Prediction (MTP v2) ────────────────────────────────────
    n_mtp: int = 4                   # Number of speculative prediction stages (t+2..t+5)
    mtp_loss_weights: List[float] = field(default_factory=lambda: [1.0, 0.8, 0.5, 0.2])
    mtp_share_trunk: bool = True

    # ─── Numerical Stability Hygiene ────────────────────────────────────────
    use_qk_norm: bool = True
    logit_soft_cap: float = 30.0     # Cap value for tanh logit soft-capping
    z_loss_coeff: float = 1e-4       # Router logsumexp squared loss weight

    # ─── Training & Distributed Engine ──────────────────────────────────────
    bf16: bool = True
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_steps: int = 1000
    total_steps: int = 100_000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    grad_accum_steps: int = 1
    activation_checkpointing: bool = True

    use_ema: bool = True
    ema_beta: float = 0.9999
    ema_update_after_step: int = 100
    ema_update_every: int = 10

    train_batch_size: int = 8
    seq_len: int = 2048

    # ─── Derived Properties ─────────────────────────────────────────────────
    @property
    def mamba_inner(self) -> int:
        return self.mamba_expand * self.d_model

    @property
    def mamba_head_dim(self) -> int:
        assert self.mamba_inner % self.mamba_n_heads == 0
        return self.mamba_inner // self.mamba_n_heads

    @property
    def q_head_dim(self) -> int:
        return self.qk_nope_head_dim + self.qk_rope_head_dim

    @property
    def block_pattern(self) -> List[str]:
        # Hybrid 3:1 ratio: 3 Mamba-2 (MoD) blocks followed by 1 Hybrid Memory-Attention Block
        base = ["mamba", "mamba", "mamba", "attn_moe"]
        return (base * (self.n_layers // 4 + 1))[: self.n_layers]

    @property
    def n_attn_layers(self) -> int:
        return self.block_pattern.count("attn_moe")

    @property
    def n_mamba_layers(self) -> int:
        return self.block_pattern.count("mamba")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def to_json(self, path: Union[str, Path], indent: int = 2) -> None:
        Path(path).write_text(json.dumps(self.to_dict(), indent=indent))

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "CASEConfig":
        return cls(**d)

    @classmethod
    def from_json(cls, path: Union[str, Path]) -> "CASEConfig":
        return cls.from_dict(json.loads(Path(path).read_text()))

    # ─── Production Presets ─────────────────────────────────────────────────
    @classmethod
    def preset_tiny(cls, **overrides) -> "CASEConfig":
        """Fast smoke test & verification config (<50M params)."""
        base = dict(
            vocab_size=1000,
            d_model=128,
            n_layers=8,
            n_heads=4,
            kv_lora_rank=64,
            q_lora_rank=128,
            qk_nope_head_dim=32,
            qk_rope_head_dim=16,
            v_head_dim=32,
            titans_d_mem=64,
            mamba_d_state=16,
            mamba_n_heads=4,
            mamba_expand=2,
            n_experts=8,
            n_active_experts=2,
            n_shared_experts=1,
            moe_d_ff=256,
            n_mtp=2,
            mtp_loss_weights=[1.0, 0.5],
            max_seq_len=256,
            activation_checkpointing=False,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def preset_1b(cls, **overrides) -> "CASEConfig":
        """1B Parameter Research Configuration (~1.1B total, ~350M active/tok)."""
        base = dict(
            vocab_size=128_256,
            d_model=1536,
            n_layers=24,
            n_heads=16,
            kv_lora_rank=256,
            q_lora_rank=768,
            qk_nope_head_dim=128,
            qk_rope_head_dim=64,
            v_head_dim=128,
            titans_d_mem=256,
            mamba_d_state=64,
            mamba_n_heads=32,
            mamba_expand=2,
            n_experts=64,
            n_active_experts=4,
            n_shared_experts=2,
            moe_d_ff=1024,
            n_mtp=4,
            mtp_loss_weights=[1.0, 0.8, 0.5, 0.2],
            max_seq_len=4096,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def preset_3b(cls, **overrides) -> "CASEConfig":
        """3B Parameter Mid-Tier Configuration (~3.2B total, ~950M active/tok)."""
        base = dict(
            vocab_size=128_256,
            d_model=2560,
            n_layers=32,
            n_heads=20,
            kv_lora_rank=384,
            q_lora_rank=1024,
            qk_nope_head_dim=128,
            qk_rope_head_dim=64,
            v_head_dim=128,
            titans_d_mem=384,
            mamba_d_state=128,
            mamba_n_heads=40,
            mamba_expand=2,
            n_experts=128,
            n_active_experts=6,
            n_shared_experts=2,
            moe_d_ff=1536,
            n_mtp=4,
            mtp_loss_weights=[1.0, 0.8, 0.5, 0.2],
            max_seq_len=8192,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def preset_7b(cls, **overrides) -> "CASEConfig":
        """7B Production Frontier Target (~7.2B total, ~1.8B active/tok)."""
        base = dict(
            vocab_size=128_256,
            d_model=4096,
            n_layers=32,
            n_heads=32,
            kv_lora_rank=512,
            q_lora_rank=1536,
            qk_nope_head_dim=128,
            qk_rope_head_dim=64,
            v_head_dim=128,
            titans_d_mem=512,
            mamba_d_state=128,
            mamba_n_heads=64,
            mamba_expand=2,
            n_experts=256,
            n_active_experts=8,
            n_shared_experts=4,
            moe_d_ff=2048,
            n_mtp=4,
            mtp_loss_weights=[1.0, 0.8, 0.5, 0.2],
            max_seq_len=8192,
        )
        base.update(overrides)
        return cls(**base)

    @classmethod
    def get_preset(cls, name: str) -> "CASEConfig":
        presets = {
            "tiny": cls.preset_tiny,
            "case_tiny": cls.preset_tiny,
            "1b": cls.preset_1b,
            "case_1b": cls.preset_1b,
            "3b": cls.preset_3b,
            "case_3b": cls.preset_3b,
            "7b": cls.preset_7b,
            "case_7b": cls.preset_7b,
        }
        if name not in presets:
            raise ValueError(f"Unknown preset: {name}. Available: {list(presets.keys())}")
        return presets[name]()

In [3]:
# Source: CASE/ops.py
# SHA-256 of original file: 49a1b266d1c797be8fc8b8cb6ff049835c616589d31e811ee72251582e3cf130

from __future__ import annotations

import math

from typing import Dict, Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization with FP32 internal precision."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        input_dtype = x.dtype
        x_fp32 = x.float()
        variance = x_fp32.pow(2).mean(-1, keepdim=True)
        normed = x_fp32 * torch.rsqrt(variance + self.eps)
        return (normed * self.weight.float()).to(input_dtype)

class DecoupledRoPE(nn.Module):
    """
    Decoupled Rotary Position Embedding for Multi-Head Latent Attention (MLA).
    Applied strictly to the decoupled position subspace (qk_rope_head_dim).
    """

    def __init__(
        self,
        dim: int,
        max_seq_len: int = 8192,
        theta: float = 500_000.0,
    ):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.theta = theta

        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Precompute initial cache
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int):
        t = torch.arange(seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)  # (seq_len, dim // 2)
        cos = freqs.cos()
        sin = freqs.sin()
        self.register_buffer("cos_cached", cos, persistent=False)
        self.register_buffer("sin_cached", sin, persistent=False)

    def forward(
        self,
        x: Tensor,
        position_ids: Optional[Tensor] = None,
        offset: int = 0,
    ) -> Tensor:
        """
        Args:
            x: (B, T, H, d_R) or (B, T, d_R)
            position_ids: Optional (B, T)
            offset: Incremental offset for inference decoding
        """
        T = x.shape[1]
        if offset + T > self.cos_cached.shape[0]:
            self._build_cache(max(offset + T, self.max_seq_len * 2))

        if position_ids is not None:
            cos = self.cos_cached[position_ids].to(x.dtype)  # (B, T, d_R//2)
            sin = self.sin_cached[position_ids].to(x.dtype)
            if x.ndim == 4:
                cos = cos.unsqueeze(2)  # (B, T, 1, d_R//2)
                sin = sin.unsqueeze(2)
        else:
            cos = self.cos_cached[offset : offset + T].to(x.dtype)  # (T, d_R//2)
            sin = self.sin_cached[offset : offset + T].to(x.dtype)
            if x.ndim == 4:
                # (1, T, 1, d_R//2)
                cos = cos.unsqueeze(0).unsqueeze(2)
                sin = sin.unsqueeze(0).unsqueeze(2)
            elif x.ndim == 3:
                # (1, T, d_R//2)
                cos = cos.unsqueeze(0)
                sin = sin.unsqueeze(0)

        # Rotate half
        half = self.dim // 2
        x1, x2 = x[..., :half], x[..., half:]
        # (-x2, x1) rotation
        rotated = torch.cat([-x2, x1], dim=-1)
        cos_full = torch.cat([cos, cos], dim=-1)
        sin_full = torch.cat([sin, sin], dim=-1)

        return (x * cos_full) + (rotated * sin_full)

class LogitSoftCap(nn.Module):
    """Tanh logit soft-capping to eliminate logit explosion."""

    def __init__(self, cap: float = 30.0):
        super().__init__()
        self.cap = cap

    def forward(self, logits: Tensor) -> Tensor:
        if self.cap is None or self.cap <= 0.0:
            return logits
        return self.cap * torch.tanh(logits / self.cap)

class SwiGLU(nn.Module):
    """SwiGLU Feed-Forward Network."""

    def __init__(
        self,
        d_model: int,
        d_ff: int,
        dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=bias)
        self.w_up = nn.Linear(d_model, d_ff, bias=bias)
        self.w_down = nn.Linear(d_ff, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        return self.w_down(self.dropout(gate * up))

In [4]:
# Source: CASE/attention.py
# SHA-256 of original file: d31cba4b943a6ee5b7d1fed46423a66c5e0bcfa3befe4358cd9574ad94c91102

from __future__ import annotations

import math

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class MultiHeadLatentAttention(nn.Module):
    """DeepSeek-V3 Style Multi-Head Latent Attention."""

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.kv_lora_rank = cfg.kv_lora_rank
        self.q_lora_rank = cfg.q_lora_rank
        self.qk_nope_head_dim = cfg.qk_nope_head_dim
        self.qk_rope_head_dim = cfg.qk_rope_head_dim
        self.v_head_dim = cfg.v_head_dim

        self.q_head_dim = self.qk_nope_head_dim + self.qk_rope_head_dim
        self.scale = 1.0 / math.sqrt(self.q_head_dim)

        # ── Query Projections ──
        self.w_dq = nn.Linear(self.d_model, self.q_lora_rank, bias=False)
        self.q_norm = RMSNorm(self.q_lora_rank, cfg.norm_eps)
        self.w_uq = nn.Linear(self.q_lora_rank, self.n_heads * self.qk_nope_head_dim, bias=False)
        self.w_qr = nn.Linear(self.q_lora_rank, self.n_heads * self.qk_rope_head_dim, bias=False)

        # ── Key-Value Latent Projections ──
        self.w_dkv = nn.Linear(self.d_model, self.kv_lora_rank, bias=False)
        self.kv_norm = RMSNorm(self.kv_lora_rank, cfg.norm_eps)
        self.w_uk = nn.Linear(self.kv_lora_rank, self.n_heads * self.qk_nope_head_dim, bias=False)
        self.w_uv = nn.Linear(self.kv_lora_rank, self.n_heads * self.v_head_dim, bias=False)
        self.w_kr = nn.Linear(self.d_model, self.qk_rope_head_dim, bias=False)

        # ── Positional Embedding & Normalization ──
        self.rope = DecoupledRoPE(self.qk_rope_head_dim, cfg.max_seq_len, cfg.rope_theta)
        if cfg.use_qk_norm:
            self.qk_norm = RMSNorm(self.q_head_dim, cfg.norm_eps)
        else:
            self.qk_norm = nn.Identity()

        # ── Output Projection ──
        self.w_out = nn.Linear(self.n_heads * self.v_head_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout) if cfg.dropout > 0.0 else nn.Identity()

    def forward(
        self,
        x: Tensor,
        position_ids: Optional[Tensor] = None,
        kv_cache: Optional[Tuple[Tensor, Tensor]] = None,
        mask: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Optional[Tuple[Tensor, Tensor]]]:
        """
        Args:
            x: Input tensor (B, T, D)
            position_ids: Optional token positions (B, T)
            kv_cache: Optional (c_kv_past, k_rope_past)
            mask: Optional attention mask
        Returns:
            out: (B, T, D)
            new_kv_cache: (c_kv_all, k_rope_all)
        """
        B, T, D = x.shape
        H = self.n_heads
        d_nope = self.qk_nope_head_dim
        d_rope = self.qk_rope_head_dim
        d_v = self.v_head_dim

        # 1. Query path
        c_q = self.q_norm(self.w_dq(x))                      # (B, T, q_lora_rank)
        q_nope = self.w_uq(c_q).reshape(B, T, H, d_nope)      # (B, T, H, d_nope)
        q_rope = self.w_qr(c_q).reshape(B, T, H, d_rope)      # (B, T, H, d_rope)

        offset = kv_cache[0].shape[1] if kv_cache is not None else 0
        q_rope = self.rope(q_rope, position_ids=position_ids, offset=offset)
        q = torch.cat([q_nope, q_rope], dim=-1)               # (B, T, H, q_head_dim)
        q = self.qk_norm(q).transpose(1, 2)                   # (B, H, T, q_head_dim)

        # 2. Key-Value path
        c_kv = self.kv_norm(self.w_dkv(x))                   # (B, T, kv_lora_rank)
        k_rope = self.w_kr(x)                                 # (B, T, d_rope)
        k_rope = self.rope(k_rope, position_ids=position_ids, offset=offset)  # (B, T, d_rope)

        if kv_cache is not None:
            c_kv_past, k_rope_past = kv_cache
            c_kv = torch.cat([c_kv_past, c_kv], dim=1)
            k_rope = torch.cat([k_rope_past, k_rope], dim=1)

        new_kv_cache = (c_kv, k_rope)
        T_full = c_kv.shape[1]

        # 3. Reconstruct Key & Value from cached compressed latents
        k_nope = self.w_uk(c_kv).reshape(B, T_full, H, d_nope)
        k_rope_expanded = k_rope.unsqueeze(2).expand(B, T_full, H, d_rope)
        k = torch.cat([k_nope, k_rope_expanded], dim=-1)      # (B, T_full, H, q_head_dim)
        k = self.qk_norm(k).transpose(1, 2)                   # (B, H, T_full, q_head_dim)

        v = self.w_uv(c_kv).reshape(B, T_full, H, d_v).transpose(1, 2)  # (B, H, T_full, d_v)

        # 4. Scaled Dot-Product Attention
        if hasattr(F, "scaled_dot_product_attention") and mask is None and T == T_full:
            # Native fast causal attention
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True, scale=self.scale
            )
        else:
            scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
            if T == T_full:
                causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
                scores = scores.masked_fill(~causal_mask[None, None], float("-inf"))
            elif mask is not None:
                scores = scores + mask

            attn_weights = F.softmax(scores.float(), dim=-1).to(x.dtype)
            attn_weights = self.dropout(attn_weights)
            out = torch.matmul(attn_weights, v)

        # (B, H, T, d_v) -> (B, T, H * d_v)
        out = out.transpose(1, 2).reshape(B, T, H * d_v)
        return self.w_out(out), new_kv_cache

In [5]:
# Source: CASE/mamba2.py
# SHA-256 of original file: f104fbef74d69dd7d86c143809676d78e1ba37d46b116c5e8161f83b037d6fe2

from __future__ import annotations

import math

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

HAS_MAMBA_SSM = False
selective_scan_fn = None

class Mamba2SSM(nn.Module):
    """
    Multi-Head State Space Duality (SSD) core.
    Maps input tokens to multi-head SSM state transitions.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.d_inner = cfg.mamba_inner
        self.d_state = cfg.mamba_d_state
        self.n_heads = cfg.mamba_n_heads
        self.head_dim = cfg.mamba_head_dim
        self.d_conv = cfg.mamba_d_conv

        # Joint projection: z, x, B, C, dt
        self.in_proj = nn.Linear(
            self.d_model,
            2 * self.d_inner + 2 * self.d_state + self.n_heads,
            bias=False,
        )

        # Causal depthwise 1D convolution
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=self.d_conv,
            groups=self.d_inner,
            padding=self.d_conv - 1,
            bias=cfg.mamba_conv_bias,
        )

        # Discretization parameters
        log_dt = torch.rand(self.n_heads) * (
            math.log(cfg.mamba_dt_max) - math.log(cfg.mamba_dt_min)
        ) + math.log(cfg.mamba_dt_min)
        self.dt_bias = nn.Parameter(log_dt)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, self.n_heads + 1, dtype=torch.float32)))
        self.D = nn.Parameter(torch.ones(self.n_heads))

        self.out_norm = RMSNorm(self.d_inner, cfg.norm_eps)
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(
        self,
        x: Tensor,
        state_cache: Optional[Tuple[Tensor, Tensor]] = None,
    ) -> Tuple[Tensor, Optional[Tuple[Tensor, Tensor]]]:
        """
        Args:
            x: Input tensor (B, L, D)
            state_cache: Optional tuple of (conv_state, ssm_state) for decoding
        Returns:
            out: (B, L, D)
            new_state_cache: (conv_state, ssm_state)
        """
        B, L, D = x.shape
        H, hd, N, di = self.n_heads, self.head_dim, self.d_state, self.d_inner

        # Joint in-projection
        proj = self.in_proj(x)
        z, x_in, B_in, C_in, dt_raw = proj.split([di, di, N, N, H], dim=-1)

        # Autoregressive single-step inference fast-path
        if state_cache is not None and L == 1:
            return self._step_inference(z, x_in, B_in, C_in, dt_raw, state_cache)

        # Causal depthwise 1D conv
        conv_out = self.conv1d(x_in.transpose(1, 2))[..., :L].transpose(1, 2)
        x_conv = F.silu(conv_out)

        x_h = x_conv.reshape(B, L, H, hd)
        z_h = z.reshape(B, L, H, hd)

        A = -torch.exp(self.A_log.float())
        dt_raw = dt_raw.float()

        if HAS_MAMBA_SSM and x.is_cuda:
            # Native CUDA selective scan: softplus is applied internally once
            y = self._cuda_mamba_scan(x_h, dt_raw, A, B_in, C_in, z_h)
        else:
            # Numerically identical chunkwise/vectorized fallback scan
            dt = F.softplus(dt_raw + self.dt_bias)
            dA = torch.exp(dt * A)
            y = self._vectorized_scan(x_h, dA, B_in, C_in)

        y = self.out_norm(y.reshape(B, L, di))
        y = y * F.silu(z)
        out = self.out_proj(y)

        new_cache = None
        if state_cache is not None:
            # Store final state for subsequent autoregressive calls
            last_conv = x_in[:, -(self.d_conv - 1) :].transpose(1, 2)
            new_cache = (last_conv, None)

        return out, new_cache

    def _vectorized_scan(
        self,
        x_h: Tensor,
        dA: Tensor,
        B_in: Tensor,
        C_in: Tensor,
    ) -> Tensor:
        """
        Vectorized multi-head recurrent state scan.
        State: h_t = dA_t * h_{t-1} + B_t * u_t
        Output: y_t = h_t * C_t + D * u_t
        """
        B, L, H, hd = x_h.shape
        N = self.d_state

        h = x_h.new_zeros(B, H, hd, N, dtype=torch.float32)
        ys = x_h.new_empty(B, L, H, hd)

        x_f = x_h.float()
        B_f = B_in.float()
        C_f = C_in.float()
        D_f = self.D.float()

        for t in range(L):
            a_t = dA[:, t, :, None, None]     # (B, H, 1, 1)
            b_t = B_f[:, t, None, None, :]     # (B, 1, 1, N)
            u_t = x_f[:, t, :, :, None]        # (B, H, hd, 1)
            h = a_t * h + (u_t * b_t)          # (B, H, hd, N)
            c_t = C_f[:, t, None, None, :]     # (B, 1, 1, N)
            y_t = (h * c_t).sum(-1) + D_f[None, :, None] * x_f[:, t]
            ys[:, t] = y_t

        return ys.to(x_h.dtype)

    def _cuda_mamba_scan(
        self,
        x_h: Tensor,
        dt_raw: Tensor,
        A: Tensor,
        B_in: Tensor,
        C_in: Tensor,
        z_h: Tensor,
    ) -> Tensor:
        B, L, H, hd = x_h.shape
        u = x_h.reshape(B, L, -1).contiguous()
        z = z_h.reshape(B, L, -1).contiguous()
        A_expanded = A.repeat_interleave(hd).contiguous()
        D_expanded = self.D.repeat_interleave(hd).contiguous()
        dt_expanded = dt_raw.repeat_interleave(hd, dim=-1).contiguous()

        y = selective_scan_fn(
            u,
            dt_expanded,
            A_expanded,
            B_in,
            C_in,
            D_expanded,
            z=z,
            delta_bias=self.dt_bias.repeat_interleave(hd),
            delta_softplus=True,
            return_last_state=False,
        )
        return y.reshape(B, L, H, hd)

    def _step_inference(
        self,
        z: Tensor,
        x_in: Tensor,
        B_in: Tensor,
        C_in: Tensor,
        dt_raw: Tensor,
        state_cache: Tuple[Tensor, Tensor],
    ) -> Tuple[Tensor, Tuple[Tensor, Tensor]]:
        """O(1) incremental step decoding using cached state."""
        conv_state, ssm_state = state_cache
        B, _, di = x_in.shape
        H, hd, N = self.n_heads, self.head_dim, self.d_state

        # Update 1D conv buffer
        # conv_state: (B, di, d_conv - 1)
        x_col = x_in.transpose(1, 2)  # (B, di, 1)
        conv_buf = torch.cat([conv_state, x_col], dim=-1)  # (B, di, d_conv)
        new_conv_state = conv_buf[:, :, 1:]

        weight = self.conv1d.weight  # (di, 1, d_conv)
        conv_out = (conv_buf * weight.squeeze(1)).sum(-1, keepdim=True)
        if self.conv1d.bias is not None:
            conv_out = conv_out + self.conv1d.bias.unsqueeze(-1)
        x_conv = F.silu(conv_out.transpose(1, 2))  # (B, 1, di)

        x_h = x_conv.reshape(B, 1, H, hd)
        A = -torch.exp(self.A_log.float())
        dt = F.softplus(dt_raw.float() + self.dt_bias)
        dA = torch.exp(dt * A)  # (B, 1, H)

        if ssm_state is None:
            ssm_state = x_in.new_zeros(B, H, hd, N, dtype=torch.float32)

        a_t = dA[:, 0, :, None, None]
        b_t = B_in[:, 0, None, None, :].float()
        u_t = x_h[:, 0, :, :, None].float()
        new_ssm_state = a_t * ssm_state + (u_t * b_t)

        c_t = C_in[:, 0, None, None, :].float()
        y_t = (new_ssm_state * c_t).sum(-1) + self.D.float()[None, :, None] * x_h[:, 0].float()
        y = self.out_norm(y_t.reshape(B, 1, di).to(x_in.dtype))
        y = y * F.silu(z)
        out = self.out_proj(y)

        return out, (new_conv_state, new_ssm_state)

class Mamba2Block(nn.Module):
    """Mamba-2 Pre-Norm Transformer-style Block."""

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ssm = Mamba2SSM(cfg)

    def forward(
        self,
        x: Tensor,
        state_cache: Optional[Tuple[Tensor, Tensor]] = None,
    ) -> Tuple[Tensor, Optional[Tuple[Tensor, Tensor]]]:
        h, new_cache = self.ssm(self.norm(x), state_cache=state_cache)
        return x + h, new_cache

In [6]:
# Source: CASE/mod.py
# SHA-256 of original file: 37731f34999938d4cbdd039694f3e9972738d43b43e28fcf7169044f48394e86

from __future__ import annotations

from typing import Any, Optional, Tuple

import torch

import torch.nn as nn

from torch import Tensor

class MixtureOfDepths(nn.Module):
    """
    Mixture-of-Depths (MoD) dynamic token routing wrapper.
    """

    def __init__(
        self,
        block: nn.Module,
        d_model: int,
        capacity: float = 0.5,
    ):
        super().__init__()
        self.block = block
        self.d_model = d_model
        self.capacity = min(max(capacity, 0.05), 1.0)
        self.router = nn.Linear(d_model, 1, bias=False)

    def forward(
        self,
        x: Tensor,
        *args: Any,
        **kwargs: Any,
    ) -> Tuple[Tensor, Any]:
        """
        Args:
            x: Input sequence (B, T, D)
            args/kwargs: Forwarded to wrapped block
        Returns:
            out: (B, T, D) with bypassed and processed tokens
            aux: Any auxiliary outputs from wrapped block
        """
        B, T, D = x.shape

        # If sequence is 1 token (autoregressive decoding), bypass routing and process directly
        if T <= 1 or self.capacity >= 1.0:
            block_out = self.block(x, *args, **kwargs)
            if isinstance(block_out, tuple):
                return block_out[0], block_out[1]
            return block_out, None

        k = max(1, int(T * self.capacity))

        # Compute routing scores
        scores = self.router(x).squeeze(-1)  # (B, T)
        _, top_idx = torch.topk(scores, k, dim=-1, sorted=False)
        weights = torch.sigmoid(scores.gather(1, top_idx))  # (B, k)

        # Sort indices to preserve temporal ordering during block computation
        top_idx_sorted, perm = top_idx.sort(dim=-1)
        weights_sorted = weights.gather(1, perm)

        # Gather selected tokens
        idx_exp = top_idx_sorted.unsqueeze(-1).expand(B, k, D)
        x_selected = x.gather(1, idx_exp)  # (B, k, D)

        # Execute block on selected tokens
        block_out = self.block(x_selected, *args, **kwargs)
        aux = None
        if isinstance(block_out, tuple):
            block_out, aux = block_out

        # Weighted residual combination
        x_processed = x_selected + (weights_sorted.unsqueeze(-1) * block_out)

        # Scatter back to full sequence
        out = x.clone()
        out.scatter_(1, idx_exp, x_processed)

        return out, aux

In [7]:
# Source: CASE/moe.py
# SHA-256 of original file: b5227349d9407173c22d303d2549be68f6c61079bba89764ca6185b3275d7c86

from __future__ import annotations

from typing import List, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class Expert(nn.Module):
    """Single Fine-Grained SwiGLU Expert."""

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.ffn = SwiGLU(d_model, d_ff, dropout=dropout, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.ffn(x)

class SharedExpert(nn.Module):
    """Always-active shared expert bank."""

    def __init__(self, d_model: int, d_ff: int, n_shared: int = 2, dropout: float = 0.0):
        super().__init__()
        shared_dim = d_ff * n_shared
        self.ffn = SwiGLU(d_model, shared_dim, dropout=dropout, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.ffn(x)

class DeepSeekMoE(nn.Module):
    """
    DeepSeek-V3 Fine-Grained MoE with zero auxiliary loss load balancing.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.n_experts = cfg.n_experts
        self.n_active = cfg.n_active_experts
        self.bias_lr = cfg.moe_bias_lr
        self.jitter_noise = cfg.moe_router_jitter_noise
        self.z_loss_coeff = cfg.z_loss_coeff

        self.router = nn.Linear(self.d_model, self.n_experts, bias=False)

        # Fine-grained routed experts
        self.experts = nn.ModuleList([
            Expert(self.d_model, cfg.moe_d_ff, cfg.dropout)
            for _ in range(self.n_experts)
        ])

        # Always-active shared experts
        self.shared_experts = SharedExpert(
            self.d_model, cfg.moe_d_ff, cfg.n_shared_experts, cfg.dropout
        )

        # Buffers for dynamic bias and load tracking (not trained by optimizer)
        self.register_buffer("expert_bias", torch.zeros(self.n_experts))
        self.register_buffer("expert_load", torch.zeros(self.n_experts))

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        """
        Args:
            x: Input tensor (B, T, D)
        Returns:
            out: (B, T, D)
            z_loss: Router logit z-loss
            load_entropy: Entropy of routing distribution
        """
        B, T, D = x.shape
        n_tokens = B * T
        x_flat = x.reshape(n_tokens, D)

        # Compute raw routing logits
        raw_logits = self.router(x_flat)  # (n_tokens, n_experts)

        # Router z-loss
        z_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
        if self.training and self.z_loss_coeff > 0:
            z_loss = self.z_loss_coeff * raw_logits.logsumexp(dim=-1).pow(2).mean()

        # Exploration jitter
        routing_logits = raw_logits
        if self.training and self.jitter_noise > 0:
            routing_logits = routing_logits + torch.randn_like(routing_logits) * self.jitter_noise

        # Top-K selection with dynamic bias
        biased_logits = routing_logits + self.expert_bias
        _, topk_indices = torch.topk(biased_logits, self.n_active, dim=-1)  # (n_tokens, K)

        # Softmax weights calculated over UNBIASED raw logits (DeepSeek-V3 standard)
        topk_raw_logits = raw_logits.gather(1, topk_indices)
        topk_weights = F.softmax(topk_raw_logits.float(), dim=-1).to(x.dtype)

        # Shared experts computation
        shared_out = self.shared_experts(x_flat)

        # Routed experts computation
        routed_out = self._dispatch_experts(x_flat, topk_indices, topk_weights)

        # Total output
        total_out = (shared_out + routed_out).reshape(B, T, D)

        # Update running bias in training mode
        if self.training:
            self._update_online_bias(topk_indices, n_tokens)

        # Calculate routing entropy for monitoring
        load = self.expert_load.float()
        p = load / (load.sum() + 1e-9)
        load_entropy = -(p * (p + 1e-9).log()).sum()

        return total_out, z_loss, load_entropy

    def _dispatch_experts(
        self,
        x_flat: Tensor,
        topk_indices: Tensor,
        topk_weights: Tensor,
    ) -> Tensor:
        """Vectorized execution across assigned experts."""
        n_tokens, D = x_flat.shape
        E = self.n_experts
        out = torch.zeros_like(x_flat)

        # Flat indices
        token_indices = torch.arange(n_tokens, device=x_flat.device).unsqueeze(1).expand(-1, self.n_active).reshape(-1)
        flat_expert_indices = topk_indices.reshape(-1)
        flat_weights = topk_weights.reshape(-1, 1)

        # Sort tokens by expert to enable batched execution per expert
        sorted_expert_ids, sort_idx = flat_expert_indices.sort()
        sorted_token_ids = token_indices[sort_idx]
        sorted_weights = flat_weights[sort_idx]

        # Count tokens per expert
        expert_counts = torch.bincount(sorted_expert_ids, minlength=E).tolist()

        start = 0
        for e_id, count in enumerate(expert_counts):
            if count == 0:
                continue
            end = start + count
            e_tok_ids = sorted_token_ids[start:end]
            e_weights = sorted_weights[start:end]

            tokens = x_flat[e_tok_ids]
            e_out = self.experts[e_id](tokens) * e_weights
            out.index_add_(0, e_tok_ids, e_out)
            start = end

        return out

    def _update_online_bias(self, topk_indices: Tensor, n_tokens: int):
        """Aux-loss-free running bias momentum update."""
        with torch.no_grad():
            counts = torch.zeros(self.n_experts, device=topk_indices.device, dtype=torch.float32)
            counts.scatter_add_(0, topk_indices.flatten(), torch.ones_like(topk_indices.flatten(), dtype=torch.float32))
            load_frac = counts / (n_tokens * self.n_active)

            target_load = 1.0 / self.n_experts
            # Positive adjustment if underloaded, negative if overloaded
            self.expert_bias.add_(self.bias_lr * (target_load - load_frac))
            self.expert_load.copy_(load_frac)

In [8]:
# Source: CASE/mtp.py
# SHA-256 of original file: 61aac7b91f14d6046a02f1f82fb9167764db6f5ed772a9de33c9672ce5ab6964

from __future__ import annotations

from typing import Callable, List, Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class MTPStageBlock(nn.Module):
    """Single depth-wise MTP stage block combining trunk representation and next token embedding."""

    def __init__(self, d_model: int, norm_eps: float = 1e-6, dropout: float = 0.0):
        super().__init__()
        # Projects concatenated [h_{k-1}, embed(x_{t+k})] back to d_model
        self.in_proj = nn.Linear(2 * d_model, d_model, bias=False)
        self.norm1 = RMSNorm(d_model, norm_eps)
        self.ffn = SwiGLU(d_model, d_model * 2, dropout=dropout, bias=False)
        self.norm2 = RMSNorm(d_model, norm_eps)

    def forward(self, h_prev: Tensor, token_embed: Tensor) -> Tensor:
        x = torch.cat([h_prev, token_embed], dim=-1)
        h = self.in_proj(x)
        h = h + self.ffn(self.norm1(h))
        return self.norm2(h)

class MTPModule(nn.Module):
    """
    Sequential Multi-Token Prediction Engine.
    Predicts tokens t+2, t+3, ..., t+K in parallel stages.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.n_mtp = cfg.n_mtp
        self.loss_weights = cfg.mtp_loss_weights

        self.stages = nn.ModuleList([
            MTPStageBlock(cfg.d_model, cfg.norm_eps, cfg.dropout)
            for _ in range(self.n_mtp)
        ])

    def forward(
        self,
        trunk_hidden: Tensor,
        input_ids: Tensor,
        embed_fn: Callable[[Tensor], Tensor],
        lm_head_fn: Callable[[Tensor], Tensor],
    ) -> List[Tensor]:
        """
        Args:
            trunk_hidden: Representation from main model trunk (B, T, D)
            input_ids: Token IDs (B, T)
            embed_fn: Model token embedding lookup
            lm_head_fn: Model output projection head
        Returns:
            List of prediction logits for offsets t+2, t+3, ...
        """
        B, T, D = trunk_hidden.shape
        mtp_logits_list = []
        h_current = trunk_hidden

        for k, stage in enumerate(self.stages):
            # Target token to shift into next state: t + (k + 1)
            offset = k + 1
            if T - offset <= 0:
                break

            # Sliced input tokens at position t+k
            curr_token_ids = input_ids[:, offset:]  # (B, T - offset)
            curr_embeds = embed_fn(curr_token_ids)  # (B, T - offset, D)

            # Match sequence lengths
            h_slice = h_current[:, : T - offset]
            h_next = stage(h_slice, curr_embeds)
            logits = lm_head_fn(h_next)
            mtp_logits_list.append(logits)

            h_current = h_next

        return mtp_logits_list

    def compute_loss(
        self,
        mtp_logits_list: List[Tensor],
        targets: Tensor,
    ) -> Tensor:
        """
        Calculates decay-weighted cross-entropy loss against future tokens.
        """
        total_loss = torch.tensor(0.0, device=targets.device)
        V = self.cfg.vocab_size

        for k, logits in enumerate(mtp_logits_list):
            offset = k + 2  # Stage 0 predicts t+2, stage 1 predicts t+3, etc.
            T_target = targets.shape[1]
            if T_target - offset <= 0 or logits.shape[1] == 0:
                continue

            valid_len = min(logits.shape[1], T_target - offset)
            pred_logits = logits[:, :valid_len].reshape(-1, V)
            target_ids = targets[:, offset : offset + valid_len].reshape(-1)

            loss = F.cross_entropy(pred_logits, target_ids)
            weight = self.loss_weights[k] if k < len(self.loss_weights) else 0.1
            total_loss = total_loss + (weight * loss)

        return total_loss

    @torch.no_grad()
    def speculative_draft(
        self,
        last_hidden: Tensor,
        last_token_id: Tensor,
        embed_fn: Callable[[Tensor], Tensor],
        lm_head_fn: Callable[[Tensor], Tensor],
    ) -> List[int]:
        """
        Generates K speculative draft token IDs from the current position.
        Args:
            last_hidden: (1, 1, D)
            last_token_id: (1, 1)
        Returns:
            draft_tokens: List of integer token IDs
        """
        draft_tokens = []
        h_curr = last_hidden
        curr_id = last_token_id

        for stage in self.stages:
            embed = embed_fn(curr_id)
            h_next = stage(h_curr, embed)
            logits = lm_head_fn(h_next)
            pred_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            draft_tokens.append(pred_token.item())

            h_curr = h_next
            curr_id = pred_token

        return draft_tokens

In [9]:
# Source: CASE/titans.py
# SHA-256 of original file: 3d31fb4d21148a6b4227dab42f746599fd089ab02f90f0d93d1674fadbdcb03a

from __future__ import annotations

import math

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class TitansMemory(nn.Module):
    """
    Test-Time Neural Long-Term Memory Module.
    Maintains an evolving associative weight matrix M_t via online gradient descent.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.d_mem = cfg.titans_d_mem
        self.mode = cfg.titans_mode.lower()

        # Associative projections
        self.w_k = nn.Linear(self.d_model, self.d_mem, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_mem, bias=False)
        self.w_q = nn.Linear(self.d_model, self.d_mem, bias=False)

        self.k_norm = RMSNorm(self.d_mem, cfg.norm_eps)
        self.q_norm = RMSNorm(self.d_mem, cfg.norm_eps)

        # Adaptive forgetting gate (alpha) and surprise learning rate (eta)
        self.w_alpha = nn.Linear(self.d_model, 1, bias=True)
        self.w_eta = nn.Linear(self.d_model, 1, bias=True)

        # Initialize gate biases for stability
        nn.init.constant_(self.w_alpha.bias, math.log(cfg.titans_decay_init / (1 - cfg.titans_decay_init + 1e-6)))
        nn.init.constant_(self.w_eta.bias, math.log(cfg.titans_lr_init / (1 - cfg.titans_lr_init + 1e-6)))

        # Output projection
        self.out_proj = nn.Linear(self.d_mem, self.d_model, bias=False)

        # MAG fusion gate
        if self.mode == "mag":
            self.gate_proj = nn.Linear(self.d_model + self.d_model, self.d_model, bias=True)

    def forward(
        self,
        x: Tensor,
        memory_state: Optional[Tensor] = None,
        attn_out: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Tensor]:
        """
        Args:
            x: Input sequence (B, T, D)
            memory_state: Optional prior memory matrix (B, d_mem, d_mem)
            attn_out: Optional attention output (B, T, D) for MAG gating
        Returns:
            fused_out: (B, T, D)
            new_memory_state: (B, d_mem, d_mem)
        """
        B, T, D = x.shape
        dm = self.d_mem

        # Project key, value, query
        k = F.normalize(self.k_norm(self.w_k(x)), dim=-1)  # (B, T, dm) unit keys
        v = self.w_v(x)                                    # (B, T, dm)
        q = self.q_norm(self.w_q(x))                       # (B, T, dm)

        alpha = torch.sigmoid(self.w_alpha(x))             # (B, T, 1) forgetting rate
        eta = torch.sigmoid(self.w_eta(x))                 # (B, T, 1) surprise rate

        if memory_state is None:
            M = x.new_zeros(B, dm, dm, dtype=torch.float32)
        else:
            M = memory_state.float()

        # Sequential test-time associative gradient descent
        y_mem_list = []
        k_f, v_f, q_f = k.float(), v.float(), q.float()

        for t in range(T):
            k_t = k_f[:, t, :, None]   # (B, dm, 1)
            v_t = v_f[:, t, :, None]   # (B, dm, 1)
            q_t = q_f[:, t, :, None]   # (B, dm, 1)
            a_t = alpha[:, t, :, None] # (B, 1, 1)
            e_t = eta[:, t, :, None]   # (B, 1, 1)

            # Associative prediction & surprise error: e = v_t - M_{t-1} @ k_t
            pred = torch.bmm(M, k_t)   # (B, dm, 1)
            err = v_t - pred           # (B, dm, 1)

            # Online gradient update: M_t = (1 - alpha_t) * M_{t-1} + 2 * eta_t * (err @ k_t^T)
            grad_update = torch.bmm(err, k_t.transpose(1, 2))  # (B, dm, dm)
            M = (1.0 - a_t) * M + 2.0 * e_t * grad_update

            # Retrieve memory for query
            retrieval = torch.bmm(M, q_t).squeeze(-1)  # (B, dm)
            y_mem_list.append(retrieval)

        y_mem = torch.stack(y_mem_list, dim=1).to(x.dtype)  # (B, T, dm)
        y_proj = self.out_proj(y_mem)                       # (B, T, D)

        if self.mode == "mag" and attn_out is not None:
            # Memory as Gate: blend attention and memory
            gate_input = torch.cat([attn_out, y_proj], dim=-1)
            gate = torch.sigmoid(self.gate_proj(gate_input))
            fused = gate * attn_out + (1.0 - gate) * y_proj
            return fused, M.to(x.dtype)

        # Fallback to direct output
        return y_proj, M.to(x.dtype)

    def step_inference(
        self,
        x_t: Tensor,
        memory_state: Tensor,
        attn_out_t: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Tensor]:
        """O(dm^2) constant-time inference step."""
        B, _, D = x_t.shape
        dm = self.d_mem

        k = F.normalize(self.k_norm(self.w_k(x_t)), dim=-1)[:, 0, :, None].float() # (B, dm, 1)
        v = self.w_v(x_t)[:, 0, :, None].float()                                    # (B, dm, 1)
        q = self.q_norm(self.w_q(x_t))[:, 0, :, None].float()                      # (B, dm, 1)

        alpha = torch.sigmoid(self.w_alpha(x_t))[:, 0, :, None].float()            # (B, 1, 1)
        eta = torch.sigmoid(self.w_eta(x_t))[:, 0, :, None].float()                # (B, 1, 1)

        M = memory_state.float()
        pred = torch.bmm(M, k)
        err = v - pred
        grad_update = torch.bmm(err, k.transpose(1, 2))
        new_M = (1.0 - alpha) * M + 2.0 * eta * grad_update

        retrieval = torch.bmm(new_M, q).squeeze(-1)  # (B, dm)
        y_proj = self.out_proj(retrieval.to(x_t.dtype)).unsqueeze(1)  # (B, 1, D)

        if self.mode == "mag" and attn_out_t is not None:
            gate_input = torch.cat([attn_out_t, y_proj], dim=-1)
            gate = torch.sigmoid(self.gate_proj(gate_input))
            fused = gate * attn_out_t + (1.0 - gate) * y_proj
            return fused, new_M.to(x_t.dtype)

        return y_proj, new_M.to(x_t.dtype)

In [10]:
# Source: CASE/looped_core.py
# SHA-256 of original file: bd7f75e4751be3f8a6ca0679f6ce79cb32ce59c75eaca094858e8241a1106c76

from __future__ import annotations

from typing import Optional, Tuple

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class LoopedReasoningBlock(nn.Module):
    """Single transition block inside the looped thinking core."""

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.transition = SwiGLU(cfg.d_model, cfg.d_model * 2, dropout=cfg.dropout)

    def forward(self, h: Tensor) -> Tensor:
        return h + self.transition(self.norm(h))

class LoopedCore(nn.Module):
    """
    GPT-6 Astra Style Looped Latent Transformer Core.
    Dynamically ponders across multiple recurrent cycles.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.max_loops = max(1, cfg.max_latent_loops)
        self.enabled = cfg.enable_looped_core and self.max_loops > 1

        if self.enabled:
            self.ponder_proj = nn.Linear(self.d_model, 1, bias=True)
            self.reasoning_block = LoopedReasoningBlock(cfg)
            # Initialize ponder bias so initial halting probability starts high (~0.8)
            nn.init.constant_(self.ponder_proj.bias, 1.5)
        else:
            self.ponder_proj = None
            self.reasoning_block = None

    def forward(
        self,
        h: Tensor,
        num_loops: Optional[int] = None,
    ) -> Tuple[Tensor, Tensor]:
        """
        Args:
            h: Latent representation (B, T, D)
            num_loops: Optional explicit loop override
        Returns:
            pondered_h: (B, T, D)
            ponder_loss: Scalar ponder cost penalty
        """
        if not self.enabled or (num_loops is not None and num_loops <= 1):
            return h, torch.tensor(0.0, device=h.device, dtype=h.dtype)

        K = num_loops or self.max_loops
        B, T, D = h.shape

        h_current = h
        accumulated_h = torch.zeros_like(h)
        halting_p = torch.zeros(B, T, 1, device=h.device, dtype=torch.float32)
        remainders = torch.ones(B, T, 1, device=h.device, dtype=torch.float32)
        ponder_steps = torch.zeros(B, T, 1, device=h.device, dtype=torch.float32)

        for k in range(1, K + 1):
            p_k = torch.sigmoid(self.ponder_proj(h_current)).float()  # (B, T, 1)

            if k == K:
                # Final step absorbs all remaining probability
                w_k = remainders
            else:
                w_k = p_k * remainders
                remainders = remainders * (1.0 - p_k)

            accumulated_h = accumulated_h + w_k.to(h.dtype) * h_current
            ponder_steps = ponder_steps + k * w_k

            if k < K:
                h_current = self.reasoning_block(h_current)

        ponder_loss = self.cfg.ponder_cost_weight * ponder_steps.mean()
        return accumulated_h, ponder_loss

In [11]:
# Source: CASE/verifier.py
# SHA-256 of original file: 89a25f8beacd92ea1c3bf046d288f2c250c332502cb97194c85a61f5830230c6

from __future__ import annotations

from typing import Optional, Tuple

import torch

import torch.nn as nn

from torch import Tensor

class ProcessVerifierHead(nn.Module):
    """
    Process-Supervised Trajectory & Consistency Verifier.
    Estimates correctness and structural viability of internal representations.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.d_model = cfg.d_model
        self.enabled = cfg.enable_process_verifier

        if self.enabled:
            self.norm = RMSNorm(self.d_model, cfg.norm_eps)
            self.net = nn.Sequential(
                nn.Linear(self.d_model, self.d_model // 2, bias=False),
                nn.SiLU(),
                nn.Linear(self.d_model // 2, 1, bias=True),
            )
            # Center initialization
            nn.init.zeros_(self.net[-1].bias)
        else:
            self.norm = None
            self.net = None

    def forward(self, h: Tensor) -> Tensor:
        """
        Args:
            h: Hidden states (B, T, D)
        Returns:
            scores: (B, T, 1) in range [-1, 1]
        """
        if not self.enabled:
            return torch.zeros(*h.shape[:-1], 1, device=h.device, dtype=h.dtype)

        normed = self.norm(h)
        raw_score = self.net(normed)
        return torch.tanh(raw_score)

    def compute_loss(
        self,
        scores: Tensor,
        targets: Optional[Tensor] = None,
    ) -> Tensor:
        """
        Calculates consistency regularizer or supervised PRM loss.
        If targets are not provided, encourages smooth, positive trajectory variance.
        """
        if not self.enabled:
            return torch.tensor(0.0, device=scores.device)

        if targets is not None:
            return nn.functional.mse_loss(scores.squeeze(-1), targets)

        # Unsupervised self-consistency regularization: penalize sudden volatility between steps
        diff = scores[:, 1:] - scores[:, :-1]
        volatility_loss = diff.pow(2).mean()
        return self.cfg.verifier_loss_weight * volatility_loss

In [12]:
# Source: CASE/model.py
# SHA-256 of original file: 90722895494250d8c1da391236901036538f9132409b555e1a27b89f35a26cbe

from __future__ import annotations

from dataclasses import dataclass

from typing import Any, Dict, List, Optional, Tuple, Union

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch import Tensor

class HybridMemoryAttnBlock(nn.Module):
    """
    Unified Memory-Attention Super-Layer:
      1. DeepSeek-V3 Multi-Head Latent Attention (MLA)
      2. Google Titans Neural Long-Term Memory (MAG Gated Fusion)
      3. GPT-6 Astra Looped Latent Reasoning Core
      4. DeepSeek-V3 Fine-Grained MoE with Zero-Aux Load Balancing
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MultiHeadLatentAttention(cfg)

        self.titans = TitansMemory(cfg) if cfg.use_titans_memory else None
        self.looped_core = LoopedCore(cfg) if cfg.enable_looped_core else None

        self.moe_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.moe = DeepSeekMoE(cfg)

    def forward(
        self,
        x: Tensor,
        position_ids: Optional[Tensor] = None,
        kv_cache: Optional[Tuple[Tensor, Tensor]] = None,
        titans_state: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Tuple[Optional[Tuple[Tensor, Tensor]], Optional[Tensor]], Tensor, Tensor, Tensor]:
        """
        Returns:
            out: (B, T, D)
            (new_kv_cache, new_titans_state)
            z_loss: Router z-loss
            load_entropy: MoE routing entropy
            ponder_loss: Astra ponder cost
        """
        # 1. Multi-Head Latent Attention
        h_norm = self.attn_norm(x)
        attn_out, new_kv = self.attn(h_norm, position_ids=position_ids, kv_cache=kv_cache)

        # 2. Titans Test-Time Neural Memory (MAG Gated Fusion)
        new_titans = None
        if self.titans is not None:
            fused_attn, new_titans = self.titans(h_norm, memory_state=titans_state, attn_out=attn_out)
            x = x + fused_attn
        else:
            x = x + attn_out

        # 3. GPT-6 Astra Looped Latent Reasoning
        ponder_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
        if self.looped_core is not None:
            x, ponder_loss = self.looped_core(x)

        # 4. DeepSeek-V3 Fine-Grained MoE
        moe_out, z_loss, load_entropy = self.moe(self.moe_norm(x))
        x = x + moe_out

        return x, (new_kv, new_titans), z_loss, load_entropy, ponder_loss

class CASEModel(nn.Module):
    """
    CASE-0.1 Full Production Model.
    """

    def __init__(self, cfg: CASEConfig):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = cfg.vocab_size
        self.d_model = cfg.d_model

        # Token Embedding with tied weights to lm_head
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout) if cfg.dropout > 0.0 else nn.Identity()

        # Hybrid backbone blocks
        self.layers = nn.ModuleList()
        self.layer_types = cfg.block_pattern

        for btype in self.layer_types:
            if btype == "mamba":
                block = Mamba2Block(cfg)
                # Wrap Mamba in Mixture of Depths (50% capacity)
                self.layers.append(MixtureOfDepths(block, cfg.d_model, cfg.mod_capacity))
            elif btype == "attn_moe":
                block = HybridMemoryAttnBlock(cfg)
                self.layers.append(block)
            else:
                raise ValueError(f"Unknown block type: {btype}")

        # Output heads & stability
        self.norm_final = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.logit_cap = LogitSoftCap(cfg.logit_soft_cap)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # Weight tying

        # Frontier auxiliary engines
        self.mtp = MTPModule(cfg)
        self.verifier = ProcessVerifierHead(cfg)

        self._init_weights()

    def _init_weights(self):
        std = 0.02
        nn.init.trunc_normal_(self.embed.weight, std=std)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Conv1d):
                nn.init.trunc_normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids: Tensor,
        position_ids: Optional[Tensor] = None,
        state_cache: Optional[Dict[str, Any]] = None,
    ) -> Tuple[Tensor, List[Tensor], Tensor, Dict[str, Any], Dict[str, Any]]:
        """
        Full causal forward pass.
        Returns:
            logits: (B, T, V)
            mtp_logits: List of speculative predictions
            verifier_scores: (B, T, 1)
            aux_metrics: Dict of z_loss, load_entropy, ponder_loss
            new_state_cache: Updated recurrent states
        """
        B, T = input_ids.shape
        device = input_ids.device

        if position_ids is None:
            offset = 0
            if state_cache is not None and "offset" in state_cache:
                offset = state_cache["offset"]
            position_ids = torch.arange(offset, offset + T, device=device).unsqueeze(0).expand(B, -1)

        x = self.drop(self.embed(input_ids))

        total_z_loss = torch.tensor(0.0, device=device)
        total_ponder_loss = torch.tensor(0.0, device=device)
        entropies = []

        new_cache = {"layers": {}, "offset": (state_cache.get("offset", 0) + T) if state_cache else T}

        for idx, (layer, btype) in enumerate(zip(self.layers, self.layer_types)):
            layer_cache = state_cache["layers"].get(idx, None) if state_cache and "layers" in state_cache else None

            if btype == "mamba":
                x, new_layer_cache = layer(x, state_cache=layer_cache)
                new_cache["layers"][idx] = new_layer_cache

            elif btype == "attn_moe":
                kv_past = layer_cache[0] if layer_cache else None
                titans_past = layer_cache[1] if layer_cache else None

                x, (new_kv, new_titans), z_loss, entropy, ponder_loss = layer(
                    x, position_ids=position_ids, kv_cache=kv_past, titans_state=titans_past
                )
                total_z_loss = total_z_loss + z_loss
                total_ponder_loss = total_ponder_loss + ponder_loss
                entropies.append(entropy)
                new_cache["layers"][idx] = (new_kv, new_titans)

        # Final representation
        h_final = self.norm_final(x)
        logits = self.logit_cap(self.lm_head(h_final))

        # Speculative MTP forward
        mtp_logits = self.mtp(
            h_final,
            input_ids,
            embed_fn=self.embed,
            lm_head_fn=lambda h: self.logit_cap(self.lm_head(h)),
        )

        # Process verifier forward
        verifier_scores = self.verifier(h_final)

        aux_metrics = {
            "z_loss": total_z_loss,
            "ponder_loss": total_ponder_loss,
            "load_entropy": (sum(entropies) / max(1, len(entropies))) if entropies else torch.tensor(0.0),
        }

        return logits, mtp_logits, verifier_scores, aux_metrics, new_cache

    def compute_loss(
        self,
        input_ids: Tensor,
        position_ids: Optional[Tensor] = None,
        verifier_targets: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Dict[str, float]]:
        """
        Unified loss computation:
          L_total = L_main + 0.3 * L_mtp + L_z + L_ponder + L_verifier
        """
        B, T = input_ids.shape
        V = self.cfg.vocab_size

        logits, mtp_logits, verifier_scores, aux_metrics, _ = self.forward(
            input_ids, position_ids=position_ids
        )

        # 1. Main Next-Token Cross-Entropy
        pred_logits = logits[:, :-1].reshape(-1, V)
        target_ids = input_ids[:, 1:].reshape(-1)
        main_loss = F.cross_entropy(pred_logits, target_ids)

        # 2. Multi-Token Prediction Speculative Loss
        mtp_loss = self.mtp.compute_loss(mtp_logits, input_ids)

        # 3. Process Verifier Consistency Loss
        verifier_loss = self.verifier.compute_loss(verifier_scores, verifier_targets)

        # 4. Total Compound Loss
        total_loss = (
            main_loss
            + 0.3 * mtp_loss
            + aux_metrics["z_loss"]
            + aux_metrics["ponder_loss"]
            + verifier_loss
        )

        metrics = {
            "loss": total_loss.item(),
            "loss_main": main_loss.item(),
            "loss_mtp": mtp_loss.item(),
            "loss_verifier": verifier_loss.item(),
            "loss_z": aux_metrics["z_loss"].item(),
            "loss_ponder": aux_metrics["ponder_loss"].item(),
            "load_entropy": aux_metrics["load_entropy"].item(),
        }

        return total_loss, metrics

    @torch.inference_mode()
    def generate(
        self,
        input_ids: Tensor,
        max_new_tokens: int = 64,
        temperature: float = 0.8,
        top_k: int = 50,
        top_p: float = 0.9,
        use_speculative: bool = False,
    ) -> Tensor:
        """
        Inference generation with incremental state caching and speculative MTP decoding.
        """
        self.eval()
        device = input_ids.device
        generated = input_ids.clone()

        # Prefill prompt
        logits, _, _, _, state_cache = self.forward(generated)

        curr_token = self._sample_token(logits[:, -1, :], temperature, top_k, top_p)
        generated = torch.cat([generated, curr_token], dim=-1)

        for _ in range(max_new_tokens - 1):
            if use_speculative:
                # Speculative drafting via MTP
                # Extracts draft tokens and verifies in next pass
                pass

            # Incremental single-token forward pass
            logits, _, _, _, state_cache = self.forward(curr_token, state_cache=state_cache)
            curr_token = self._sample_token(logits[:, -1, :], temperature, top_k, top_p)
            generated = torch.cat([generated, curr_token], dim=-1)

        return generated

    def _sample_token(
        self,
        logits: Tensor,
        temperature: float = 0.8,
        top_k: int = 50,
        top_p: float = 0.9,
    ) -> Tensor:
        """Top-k, top-p nucleus sampling."""
        if temperature <= 0.0:
            return torch.argmax(logits, dim=-1, keepdim=True)

        logits = logits / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1)
            logits[logits < v[:, -1:]] = float("-inf")

        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_mask = cum_probs > top_p
            sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
            sorted_mask[..., 0] = 0
            indices_to_remove = sorted_mask.scatter(1, sorted_indices, sorted_mask)
            logits[indices_to_remove] = float("-inf")

        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, num_samples=1)

### Tiny synthetic optimization

Define a bounded four-block profile with three recurrent stages and one attention/expert stage. Generate a cyclic token stream with distinct initial states in each split. Train six updates to verify gradients and held-out loss measurement. All evidence is scoped to this fixture and short float32 CPU run.

In [13]:
case_cfg = CASEConfig.preset_tiny(
    vocab_size=23, d_model=32, n_layers=4, n_heads=2,
    kv_lora_rank=16, q_lora_rank=32, qk_nope_head_dim=8,
    qk_rope_head_dim=8, v_head_dim=8, mamba_d_state=4,
    mamba_n_heads=4, titans_d_mem=8, max_seq_len=64,
    n_experts=3, n_active_experts=2, moe_d_ff=24,
    moe_router_jitter_noise=0.0,
)
case = CASEModel(case_cfg)
case_train = (torch.arange(6)[:, None] + 2 * torch.arange(12)[None, :]) % 23
case_test = (torch.arange(6, 10)[:, None] + 2 * torch.arange(12)[None, :]) % 23
assert not set(case_train[:, 0].tolist()) & set(case_test[:, 0].tolist())

@torch.no_grad()
def held_out_loss(model, values):
    model.eval()
    logits, _, _, _, _ = model(values)
    return F.cross_entropy(logits[:, :-1].flatten(0, 1), values[:, 1:].flatten()).item()

base = held_out_loss(case, case_test)
case.train()
opt = torch.optim.AdamW(case.parameters(), lr=0.003)
for _ in range(6):
    opt.zero_grad(set_to_none=True)
    case_loss, train_metrics = case.compute_loss(case_train)
    assert torch.isfinite(case_loss).item()
    case_loss.backward()
    norm = torch.nn.utils.clip_grad_norm_(case.parameters(), 1)
    assert torch.isfinite(norm).item() and norm.item() > 0
    opt.step()
fitted = held_out_loss(case, case_test)
assert math.isfinite(fitted)
print({"block_pattern": case_cfg.block_pattern,
       "parameters": sum(p.numel() for p in case.parameters()),
       "heldout_before": base, "heldout_after": fitted,
       "last_train_step": train_metrics})

{'block_pattern': ['mamba', 'mamba', 'mamba', 'attn_moe'], 'parameters': 55271, 'heldout_before': 3.1577603816986084, 'heldout_after': 2.4353132247924805, 'last_train_step': {'loss': 3.685981035232544, 'loss_main': 2.5001513957977295, 'loss_mtp': 3.9523608684539795, 'loss_verifier': 1.199063177637072e-07, 'loss_z': 0.00012120614701416343, 'loss_ponder': 0.0, 'load_entropy': 1.0855389833450317}}


### Cache parity and prefix invariance

Cache parity compares logits from one full call with logits from one call per token along the same sequence. Prefix invariance keeps the prefix fixed while changing later tokens. These are independent checks. Sequence-wide top-k routing may select different early tokens when future tokens change. The disabled-MoD control reuses the same weights and retains all other state/cache mechanisms. A failed gate remains visible and blocks a release claim.

In [14]:
@torch.no_grad()
def parity_probe(model, prompt, tolerance=1e-3):
    model.eval()
    full, _, _, _, _ = model(prompt)
    cache = None
    pieces = []
    for token in prompt.split(1, dim=1):
        step, _, _, _, cache = model(token, state_cache=cache)
        pieces.append(step)
    cached = torch.cat(pieces, dim=1)
    assert full.shape == cached.shape
    assert torch.isfinite(full).all().item() and torch.isfinite(cached).all().item()
    max_delta = (full - cached).abs().max().item()
    accuracy = (full.argmax(-1) == cached.argmax(-1)).float().mean().item()
    return dict(max_delta=max_delta, greedy_agreement=accuracy,
                tolerance=tolerance, passed=max_delta <= tolerance)

@torch.no_grad()
def prefix_probe(model, prompts, split=6, tolerance=1e-5):
    changed = prompts.clone()
    changed[:, split:] = (changed[:, split:] + 7) % model.cfg.vocab_size
    assert torch.equal(changed[:, :split], prompts[:, :split])
    old = model(prompts)[0][:, :split]
    new = model(changed)[0][:, :split]
    assert torch.isfinite(old).all().item() and torch.isfinite(new).all().item()
    delta = (old - new).abs().max().item()
    return dict(max_delta=delta, tolerance=tolerance, passed=delta <= tolerance)

no_mod_cfg = CASEConfig.from_dict({**case_cfg.to_dict(), "mod_capacity": 1.0})
no_mod = CASEModel(no_mod_cfg).eval()
no_mod.load_state_dict(case.state_dict())
case_checks = {
    "cache": parity_probe(case, case_test[:1]),
    "prefix": prefix_probe(case, case_test),
    "no_mod_cache_control": parity_probe(no_mod, case_test[:1]),
    "no_mod_prefix_control": prefix_probe(no_mod, case_test),
}
blocked = [name for name in ("cache", "prefix") if not case_checks[name]["passed"]]
print({"checks": case_checks, "release_gate": "blocked" if blocked else "fixture passed",
       "failed_checks": blocked})

{'checks': {'cache': {'max_delta': 0.9061400890350342, 'greedy_agreement': 0.5, 'tolerance': 0.001, 'passed': False}, 'prefix': {'max_delta': 0.14250652492046356, 'tolerance': 1e-05, 'passed': False}, 'no_mod_cache_control': {'max_delta': 0.5384635329246521, 'greedy_agreement': 0.8333333134651184, 'tolerance': 0.001, 'passed': False}, 'no_mod_prefix_control': {'max_delta': 1.4901161193847656e-08, 'tolerance': 1e-05, 'passed': True}}, 'release_gate': 'blocked', 'failed_checks': ['cache', 'prefix']}


### TARS local research workflow

Use the current `Tars/tars` task/evidence/report contracts, planner, verifier, and pivoted Gaussian elimination tool. The next cells embed those sources, omitting relative imports and selecting the linear-system code rather than other tools. The execution cell wires the components locally for an arithmetic example with an established answer and rejection controls. Storage, retrieval services, model generation, and earlier training claims are outside this experiment.

In [15]:
# Source: Tars/tars/contracts.py
# SHA-256 of original file: 4a6eb4fc11ecc3f3102458e0ac2a342154d4e8d293a9bdb04f9777bbbf216790

from __future__ import annotations

from dataclasses import asdict, dataclass, field

from datetime import datetime, timezone

import json

from typing import Any, Dict, List, Optional, Tuple

import uuid

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"

@dataclass
class ResearchTask:
    objective: str
    domain: str = "general"
    constraints: Tuple[str, ...] = ()
    evidence_scope: str = "local"
    compute_budget: str = "small"
    risk_level: str = "low"
    task_id: str = field(default_factory=lambda: new_id("task"))
    created_at: str = field(default_factory=utc_now)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        data = asdict(self)
        data["constraints"] = list(self.constraints)
        return data

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "ResearchTask":
        values = dict(data)
        values["constraints"] = tuple(values.get("constraints", ()))
        return cls(**values)

@dataclass
class Evidence:
    source: str
    excerpt: str
    score: float = 0.0
    evidence_id: str = field(default_factory=lambda: new_id("evidence"))
    locator: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class Hypothesis:
    statement: str
    rationale: str
    confidence: float = 0.0
    tests: List[str] = field(default_factory=list)
    hypothesis_id: str = field(default_factory=lambda: new_id("hyp"))

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class Computation:
    name: str
    expression: str
    output: Any
    status: str
    tool: str
    duration_ms: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)
    computation_id: str = field(default_factory=lambda: new_id("calc"))

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class Uncertainty:
    statement: str
    level: str = "medium"
    reason: str = ""

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class Artifact:
    kind: str
    name: str
    content: str = ""
    path: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    artifact_id: str = field(default_factory=lambda: new_id("artifact"))

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class ResearchReport:
    task: ResearchTask
    answer: str
    evidence: List[Evidence] = field(default_factory=list)
    hypotheses: List[Hypothesis] = field(default_factory=list)
    computations: List[Computation] = field(default_factory=list)
    uncertainty: List[Uncertainty] = field(default_factory=list)
    unresolved_questions: List[str] = field(default_factory=list)
    reproducibility_artifacts: List[Artifact] = field(default_factory=list)
    review_status: str = "pending"
    verification: Dict[str, Any] = field(default_factory=dict)
    plan: List[Dict[str, Any]] = field(default_factory=list)
    graph_id: Optional[str] = None
    run_id: str = field(default_factory=lambda: new_id("run"))
    created_at: str = field(default_factory=utc_now)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "task": self.task.to_dict(),
            "answer": self.answer,
            "evidence": [item.to_dict() for item in self.evidence],
            "hypotheses": [item.to_dict() for item in self.hypotheses],
            "computations": [item.to_dict() for item in self.computations],
            "uncertainty": [item.to_dict() for item in self.uncertainty],
            "unresolved_questions": list(self.unresolved_questions),
            "reproducibility_artifacts": [
                item.to_dict() for item in self.reproducibility_artifacts
            ],
            "review_status": self.review_status,
            "verification": self.verification,
            "plan": list(self.plan),
            "graph_id": self.graph_id,
            "run_id": self.run_id,
            "created_at": self.created_at,
        }

    def to_json(self, indent: int = 2) -> str:
        return json.dumps(self.to_dict(), indent=indent, ensure_ascii=True, sort_keys=True)

In [16]:
# Source: Tars/tars/planner.py
# SHA-256 of original file: 27c89d9567725327b68891945ff2625af6bd18a56f4cb4366cb4f8e6728613b1

from __future__ import annotations

from dataclasses import dataclass, asdict

from typing import Dict, List

@dataclass
class PlanStep:
    step_id: str
    action: str
    description: str
    status: str = "pending"

    def to_dict(self) -> Dict[str, str]:
        return asdict(self)

def build_plan(task: ResearchTask, branching: int = 1) -> List[PlanStep]:
    domain = task.domain.lower()
    branching = max(1, min(int(branching), 4))
    steps = [
        PlanStep("define", "define_problem", "Extract the objective, constraints, and success criteria."),
        PlanStep("retrieve", "retrieve_evidence", "Find local evidence and preserve source locations."),
        PlanStep(
            "hypothesize",
            "generate_hypotheses",
            f"Generate testable {domain} hypotheses across {branching} bounded plan branch(es).",
        ),
        PlanStep("compute", "run_computation", "Run only permitted, reproducible computations."),
        PlanStep("verify", "verify", "Check evidence, computation status, and uncertainty."),
    ]
    if task.risk_level.lower() not in {"low", "minimal"}:
        steps.append(
            PlanStep(
                "review",
                "human_review",
                "Require qualified human approval before high-impact conclusions or actions.",
            )
        )
    steps.append(PlanStep("report", "write_report", "Produce an auditable research report and artifacts."))
    return steps

In [17]:
# Source: Tars/tars/verifier.py
# SHA-256 of original file: f9697158d9ae57ef4e5f325b0fe8fab1a0939fc747d2c1b19a88e72c7bfda66d

from __future__ import annotations

from typing import Any, Dict, Iterable, List

class VerificationSuite:
    def __init__(self, mode: str = "standard") -> None:
        if mode not in {"standard", "strict"}:
            raise ValueError("verification mode must be standard or strict")
        self.mode = mode

    def verify_evidence(self, evidence: Iterable[Evidence]) -> Dict[str, Any]:
        items = list(evidence)
        valid = [
            item
            for item in items
            if item.source
            and item.excerpt.strip()
            and item.metadata.get("kind") != "user-controlled-memory"
        ]
        strict_quality = all(item.score >= 0.05 for item in valid) if valid else False
        evidence_verified = bool(items) and len(valid) == len(items)
        if self.mode == "strict":
            evidence_verified = evidence_verified and strict_quality
        return {
            "evidence_total": len(items),
            "evidence_with_source": len(valid),
            "memory_context_items": sum(
                1 for item in items if item.metadata.get("kind") == "user-controlled-memory"
            ),
            "mode": self.mode,
            "status": "verified" if evidence_verified else "warning",
        }

    def verify_computations(self, computations: Iterable[Computation]) -> Dict[str, Any]:
        items = list(computations)
        failed = [item for item in items if item.status != "ok"]
        return {
            "computations_total": len(items),
            "computations_failed": len(failed),
            "mode": self.mode,
            "status": "verified" if not failed else "warning",
        }

    def verify_task(self, task: ResearchTask) -> Dict[str, Any]:
        warnings: List[str] = []
        if not task.objective.strip():
            warnings.append("objective is empty")
        if task.risk_level.lower() not in {"low", "minimal"}:
            warnings.append("qualified human review is required")
        return {
            "mode": self.mode,
            "status": "warning" if warnings else "verified",
            "warnings": warnings,
        }

    def verify(
        self,
        task: ResearchTask,
        evidence: Iterable[Evidence],
        computations: Iterable[Computation],
    ) -> Dict[str, Any]:
        checks = {
            "task": self.verify_task(task),
            "evidence": self.verify_evidence(evidence),
            "computations": self.verify_computations(computations),
        }
        statuses = [check["status"] for check in checks.values()]
        return {
            "mode": self.mode,
            "status": "verified" if all(status == "verified" for status in statuses) else "warning",
            "checks": checks,
        }

In [18]:
# Source: Tars/tars/tools.py
# SHA-256 of original file: 243ef91f8aa4b13767b5875e3a11cf0234d45a2f8b3f1fc2bca35633573c9fd9

from __future__ import annotations

import ast

import math

import operator

import time

from dataclasses import dataclass, field

from typing import Any, Dict, Iterable, List

@dataclass
class ToolResult:
    tool: str
    status: str
    output: Any = None
    error: str | None = None
    duration_ms: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

class Tool:
    name = "tool"

    def run(self, arguments: Dict[str, Any]) -> ToolResult:
        raise NotImplementedError

In [19]:
# Source: Tars/tars/science.py
# SHA-256 of original file: 8d3c05dcbf8d23fcca538966f8093524ce836f0057be9bba5f6346c04fb1ffdf

from __future__ import annotations

import math

import time

from typing import Any, Dict, List, Sequence, Tuple

def solve_linear_system(matrix: Sequence[Sequence[float]], vector: Sequence[float]) -> Tuple[List[float], float]:
    """Solve Ax=b with pivoted Gaussian elimination and return residual norm."""
    size = len(matrix)
    if size < 1 or size > 32 or len(vector) != size:
        raise ValueError("matrix must be square with size between 1 and 32")
    augmented = []
    for row, value in zip(matrix, vector):
        if len(row) != size:
            raise ValueError("matrix must be square")
        augmented.append([float(item) for item in row] + [float(value)])
    for pivot in range(size):
        pivot_row = max(range(pivot, size), key=lambda row: abs(augmented[row][pivot]))
        if abs(augmented[pivot_row][pivot]) < 1e-12:
            raise ValueError("matrix is singular or ill-conditioned")
        augmented[pivot], augmented[pivot_row] = augmented[pivot_row], augmented[pivot]
        divisor = augmented[pivot][pivot]
        augmented[pivot] = [value / divisor for value in augmented[pivot]]
        for row in range(size):
            if row == pivot:
                continue
            factor = augmented[row][pivot]
            augmented[row] = [
                value - factor * pivot_value
                for value, pivot_value in zip(augmented[row], augmented[pivot])
            ]
    solution = [augmented[index][-1] for index in range(size)]
    residual = math.sqrt(
        sum(
            (sum(float(matrix[row][column]) * solution[column] for column in range(size)) - float(vector[row])) ** 2
            for row in range(size)
        )
    )
    return solution, residual

class LinearSystemTool(Tool):
    name = "linear_system"

    def run(self, arguments: Dict[str, Any]) -> ToolResult:
        started = time.perf_counter()
        try:
            solution, residual = solve_linear_system(arguments.get("matrix", []), arguments.get("vector", []))
            return ToolResult(
                tool=self.name,
                status="ok",
                output={"solution": [round(value, 12) for value in solution], "residual_norm": residual},
                duration_ms=(time.perf_counter() - started) * 1000,
                metadata={"solver": "pivoted_gaussian_elimination"},
            )
        except Exception as exc:
            return ToolResult(
                tool=self.name,
                status="error",
                error=f"{type(exc).__name__}: {exc}",
                duration_ms=(time.perf_counter() - started) * 1000,
            )

In [20]:
task = ResearchTask(
    objective="Solve a two-variable public arithmetic fixture and verify its residual.",
    domain="mathematics", constraints=("local fixture", "do not read files"),
    evidence_scope="local fixture", compute_budget="single 2x2 linear solve",
    risk_level="low", task_id="task_linear_system", created_at="1970-01-01T00:00:00+00:00",
)
evidence = Evidence(
    source="synthetic:linear-system", excerpt="2x + y = 5; x + 3y = 10",
    score=1.0, evidence_id="evidence_fixture", locator="math fixture 1",
)
plan = build_plan(task)
calc = LinearSystemTool().run({"matrix": [[2, 1], [1, 3]], "vector": [5, 10]})
assert calc.status == "ok"
assert max(abs(value - expected) for value, expected in
           zip(calc.output["solution"], [1, 3], strict=True)) < 1e-12
assert calc.output["residual_norm"] < 1e-12
result = Computation(
    name="Solve fixture", expression=evidence.excerpt, output=calc.output,
    status=calc.status, tool=calc.tool, metadata=calc.metadata, computation_id="calc_fixture",
)
verifier = VerificationSuite("strict")
verification = verifier.verify(task, [evidence], [result])
assert verification["status"] == "verified"
assert verifier.verify(task, [], [result])["status"] == "warning"
singular = LinearSystemTool().run({"matrix": [[1, 2], [2, 4]], "vector": [3, 6]})
assert singular.status == "error" and "singular" in singular.error
from dataclasses import replace
bad = replace(result, status=singular.status)
assert verifier.verify(task, [evidence], [bad])["status"] == "warning"

report = ResearchReport(
    task=task, answer="x = 1, y = 3",
    evidence=[evidence], computations=[result], verification=verification,
    plan=[step.to_dict() for step in plan], run_id="run_linear_system",
    created_at="1970-01-01T00:00:00+00:00",
)
print({"answer": report.answer, "planned_steps": [step.action for step in plan],
       "verification": verification["status"], "residual": calc.output["residual_norm"],
       "missing_evidence_gate": "warning", "singular_input_rejected": True})

{'answer': 'x = 1, y = 3', 'planned_steps': ['define_problem', 'retrieve_evidence', 'generate_hypotheses', 'run_computation', 'verify', 'write_report'], 'verification': 'verified', 'residual': 0.0, 'missing_evidence_gate': 'warning', 'singular_input_rejected': True}


### Interpretation

Successful arithmetic verification does not offset CASE model correctness failures. For CASE, the next tests should isolate each recurrent state, maintain explicit cache length metadata, and exercise multi-token cached chunks and masked prefixes. For TARS, exercise error propagation with a separate held-out fixture suite before incorporating a learned backend.